# Silver Layer - Data Cleaning and Enrichement

## Read Bonze tables

In [0]:

taxi_trip_bronze = spark.table(
    "taxicatalog.`taxi_project-schema`.taxi_trip_bronze"
)

taxi_location_bronze = spark.table(
    "taxicatalog.`taxi_project-schema`.taxi_location_bronze"
)

print("Bronze tables loaded successfully.")
print(f"Trip records: {taxi_trip_bronze.count()}")
print(f"Location records: {taxi_location_bronze.count()}")

## Inspect the Bronze data

In [0]:

print("=== TAXI TRIP BRONZE SCHEMA ===")
taxi_trip_bronze.printSchema()

print("\n=== TAXI LOCATION BRONZE SCHEMA ===")
taxi_location_bronze.printSchema()

print("\n=== SAMPLE TRIP RECORDS ===")
display(taxi_trip_bronze.limit(5))

print("\n=== SAMPLE LOCATION RECORDS ===")
display(taxi_location_bronze.limit(5))

## Clean the trip data

We'll do the basic cleaning first:

- Remove duplicate trip_id records
- Remove rows where essential IDs/dates are missing
- Calculate trip duration
- Keep the ingestion metadata

In [0]:
from pyspark.sql.functions import (
    col,
    round,
    unix_timestamp
)

taxi_trip_silver = (
    taxi_trip_bronze

    # Remove duplicate trips
    .dropDuplicates(["trip_id"])

    # Remove records missing essential fields
    .filter(col("trip_id").isNotNull())
    .filter(col("pickup_datetime").isNotNull())
    .filter(col("dropoff_datetime").isNotNull())
    .filter(col("pickup_location_id").isNotNull())
    .filter(col("dropoff_location_id").isNotNull())

    # Calculate trip duration in minutes
    .withColumn(
        "trip_duration_minutes",
        round(
            (
                unix_timestamp(col("dropoff_datetime"))
                - unix_timestamp(col("pickup_datetime"))
            ) / 60,
            2
        )
    )
)

display(taxi_trip_silver)

## Data-quality checks

In [0]:
# Silver Layer - Cell 4
# Data quality checks

from pyspark.sql.functions import (
    col,
    sum,
    when,
    min,
    max,
    avg
)

print("=== RECORD COUNT ===")
print(f"Total records: {taxi_trip_silver.count()}")

print("\n=== NULL VALUES ===")

null_counts = taxi_trip_silver.select(
    [
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in taxi_trip_silver.columns
    ]
)

display(null_counts)

print("\n=== TRIP DURATION ===")

taxi_trip_silver.select(
    min("trip_duration_minutes").alias("min_duration"),
    max("trip_duration_minutes").alias("max_duration"),
    avg("trip_duration_minutes").alias("avg_duration")
).display()

print("\n=== TRIP DISTANCE ===")

taxi_trip_silver.select(
    min("trip_distance_miles").alias("min_distance"),
    max("trip_distance_miles").alias("max_distance"),
    avg("trip_distance_miles").alias("avg_distance")
).display()

print("\n=== FARE / TOTAL AMOUNT ===")

taxi_trip_silver.select(
    min("fare_amount").alias("min_fare"),
    max("fare_amount").alias("max_fare"),
    avg("fare_amount").alias("avg_fare"),
    min("total_amount").alias("min_total"),
    max("total_amount").alias("max_total")
).display()

## Apply data-quality rules

We'll remove records that don't make business sense:
- Trip duration must be greater than 0
- Trip distance must be greater than 0
- Fare must be greater than or equal to 0
- Total amount must be greater than or equal to 0

In [0]:
# Silver Layer - Cell 5
# Apply data-quality rules

taxi_trip_silver = (
    taxi_trip_silver
    .filter(col("trip_duration_minutes") > 0)
    .filter(col("trip_distance_miles") > 0)
    .filter(col("fare_amount") >= 0)
    .filter(col("total_amount") >= 0)
)

print(f"Records after quality filtering: {taxi_trip_silver.count()}")

display(taxi_trip_silver)

## Add useful time dimensions

In [0]:
# Silver Layer - Cell 6
# Add time-based analytical columns

from pyspark.sql.functions import (
    to_date,
    hour,
    dayofweek,
    date_format,
    when
)

taxi_trip_silver = (
    taxi_trip_silver

    .withColumn(
        "pickup_date",
        to_date(col("pickup_datetime"))
    )

    .withColumn(
        "pickup_hour",
        hour(col("pickup_datetime"))
    )

    .withColumn(
        "pickup_day_of_week",
        date_format(col("pickup_datetime"), "EEEE")
    )

    .withColumn(
        "pickup_day_number",
        dayofweek(col("pickup_datetime"))
    )

    .withColumn(
        "time_of_day",
        when(col("pickup_hour").between(6, 11), "Morning")
        .when(col("pickup_hour").between(12, 16), "Afternoon")
        .when(col("pickup_hour").between(17, 20), "Evening")
        .otherwise("Night")
    )
)

display(taxi_trip_silver)

## Join with the location table

In [0]:
# Silver Layer - Cell 7
# Join pickup and drop-off location information

pickup_location = (
    taxi_location_bronze
    .select(
        col("location_id").alias("pickup_location_id"),
        col("borough").alias("pickup_borough"),
        col("zone").alias("pickup_zone"),
        col("service_zone").alias("pickup_service_zone")
    )
)

dropoff_location = (
    taxi_location_bronze
    .select(
        col("location_id").alias("dropoff_location_id"),
        col("borough").alias("dropoff_borough"),
        col("zone").alias("dropoff_zone"),
        col("service_zone").alias("dropoff_service_zone")
    )
)

taxi_trip_silver = (
    taxi_trip_silver
    .join(
        pickup_location,
        on="pickup_location_id",
        how="left"
    )
    .join(
        dropoff_location,
        on="dropoff_location_id",
        how="left"
    )
)

display(taxi_trip_silver)

## Create the final Silver dataset

Explicitly select the columns we want in the final table. This is better than carrying every Bronze column forward.

In [0]:

taxi_trip_silver = taxi_trip_silver.select(
    "trip_id",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_day_number",
    "time_of_day",

    "pickup_location_id",
    "pickup_borough",
    "pickup_zone",
    "pickup_service_zone",

    "dropoff_location_id",
    "dropoff_borough",
    "dropoff_zone",
    "dropoff_service_zone",

    "passenger_count",
    "trip_distance_miles",
    "trip_duration_minutes",

    "fare_amount",
    "tax_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "payment_type",

    "_ingestion_timestamp",
    "_source_file"
)

display(taxi_trip_silver)

## Create Silver location table

In [0]:
# Silver Layer - Cell 9
# Create clean location dimension

taxi_location_silver = (
    taxi_location_bronze
    .dropDuplicates(["location_id"])
    .filter(col("location_id").isNotNull())
    .filter(col("borough").isNotNull())
    .filter(col("zone").isNotNull())
    .select(
        "location_id",
        "borough",
        "zone",
        "service_zone",
        "_ingestion_timestamp",
        "_source_file"
    )
)

display(taxi_location_silver)

## Write Silver tables

In [0]:

(
    taxi_trip_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "taxicatalog.`taxi_project-schema`.taxi_trip_silver"
    )
)

In [0]:

(
    taxi_location_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "taxicatalog.`taxi_project-schema`.taxi_location_silver"
    )
)

## Verify

In [0]:

spark.sql("""
SHOW TABLES IN taxicatalog.`taxi_project-schema`
""").display()

In [0]:
display(
    spark.table(
        "taxicatalog.`taxi_project-schema`.taxi_trip_silver"
    )
)